In [25]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup

import os
import numpy as np
from spikeinterface.core import concatenate_recordings

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.stats import pearsonr
import pandas as pd
import numpy as np
from matplotlib.collections import LineCollection
from probeinterface import write_probeinterface, read_probeinterface, Probe
from scipy.io import loadmat
import pickle

def load_mat(path):
    try:
        return loadmat(path)
    except NotImplementedError:
        import mat73
        return mat73.loadmat(path)

probe_data = load_mat("/media/ubuntu/sda/duan/raw_data/chanMap_DCX_5mm.mat")
probe_x = probe_data['xcoords']
probe_y = probe_data['ycoords']

probe_position = pd.DataFrame(probe_x)
probe_position[1] = probe_y
probe_position['chan_map'] = probe_data['chanMap0ind'].astype(int)

chan_map = pd.read_csv('/media/ubuntu/sda/duan/raw_data/ch_map_R.csv')
merged = chan_map.merge(probe_position, left_on='probeloc', right_on='chan_map')\
                 .iloc[chan_map.index]\
                 .reset_index(drop=True)

probe = Probe()
probe.set_contacts(positions=merged.iloc[:, 2:4])
probe.set_device_channel_indices(range(256))


In [4]:
output_dir = '/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/'

with open(output_dir + '/neuron_inf.pickle', 'rb') as f:
    neuron_inf = pickle.load(f)

gt_detect_array = pd.read_csv(output_dir + '/gt_detect_array.csv')

In [3]:
with PdfPages(f"{output_dir}/unit_histplots.pdf") as pdf:
    for unit in gt_detect_array['unit_id'].unique():
        sample_max = gt_detect_array['sample_index'].max()
        temp = gt_detect_array[gt_detect_array['unit_id'] == unit]
        plt.figure(figsize=(6, 4))
        sns.histplot(temp['sample_index'])
        plt.xlim(0, sample_max)
        plt.title(unit)
        plt.xlabel('Sample Index')
        pdf.savefig()
        plt.close()



In [5]:
rec_params = pd.read_csv("/media/ubuntu/sda/duan/result/260121/rec_params.csv")
condition = pd.read_csv("/media/ubuntu/sda/duan/result/260121/images_sequence_10000.csv")

rec_params = rec_params[rec_params['bhv_codes'] == 10]
rec_params.index = range(10000)
rec_params = pd.concat((rec_params, condition), axis = 1)
# rec_params['trial_condition'] = rec_params['trial_condition'].astype(int)

rec_params['rec_codes_points'] = rec_params['rec_codes_points'] / 3 
rec_params['rec_codes_points'] = rec_params['rec_codes_points'].astype(int)


rec_params = rec_params[rec_params['trial_error'] == 0.0]


In [6]:
from joblib import Parallel, delayed
from tqdm import tqdm

output_dir = '/home/ubuntu/Documents/jct/project/sorted/260121/groups_01_to_05_sliced/'

# Time windows in samples (at 10kHz sampling rate)
window_before_samples = 1500
window_after_samples = 4500
sample_to_ms = 0.1  # 10kHz = 0.1ms per sample
SAMPLING_RATE = 10000

# Convert to milliseconds
window_before_ms = int(window_before_samples * sample_to_ms)  # 150 ms
window_after_ms = int(window_after_samples * sample_to_ms)    # 450 ms
total_time_ms = window_before_ms + window_after_ms            # 600 ms

# Raster matrix: bin by milliseconds (1ms per bin)
n_time_bins_ms = total_time_ms  # 600 bins, each representing 1ms

# PSTH smoothing window
psth_window_size_ms = 20  # 20ms smoothing window

all_neuron_ids = sorted(gt_detect_array['unit_id'].unique())
trial_stim_points = rec_params['rec_codes_points'].astype(int).values
stimulus_ids = rec_params['image_name'].astype(str).values

stimulus_unique = pd.unique(stimulus_ids)
stimulus_id_to_index = {sid: idx for idx, sid in enumerate(stimulus_unique)}

n_trials = len(trial_stim_points)
n_neurons = len(all_neuron_ids)

# Helper function for raster matrix computation per neuron
def compute_raster_for_neuron(neuron_id, neuron_idx, gt_detect_array, trial_stim_points, 
                               window_before_samples, window_after_samples, 
                               window_before_ms, n_time_bins_ms, sample_to_ms):
    """Compute raster matrix for a single neuron (binned by milliseconds)"""
    neuron_spikes = gt_detect_array.loc[gt_detect_array['unit_id'] == neuron_id, 'sample_index'].values.astype(np.int64)
    neuron_spikes.sort()
    
    neuron_raster = np.zeros((n_trials, n_time_bins_ms), dtype=np.float32)
    
    for trial_idx, stim_point in enumerate(trial_stim_points):
        start_ext_samples = stim_point - window_before_samples
        end_ext_samples = stim_point + window_after_samples
        
        # Get spikes in the time window
        trial_spikes_samples = neuron_spikes[(neuron_spikes >= start_ext_samples) & (neuron_spikes <= end_ext_samples)]
        
        # Convert spike times to milliseconds relative to window start
        trial_spikes_ms = (trial_spikes_samples - start_ext_samples) * sample_to_ms
        
        # Bin spikes into 1ms bins
        for spike_ms in trial_spikes_ms:
            bin_idx = int(np.floor(spike_ms))
            if 0 <= bin_idx < n_time_bins_ms:
                neuron_raster[trial_idx, bin_idx] += 1
    
    return neuron_idx, neuron_raster

# Parallel computation of raster matrix
print("Computing raster matrix (parallelized, binned by milliseconds)...")
raster_results = Parallel(n_jobs=-1, backend='threading')(
    delayed(compute_raster_for_neuron)(
        neuron_id, neuron_idx, gt_detect_array, trial_stim_points,
        window_before_samples, window_after_samples,
        window_before_ms, n_time_bins_ms, sample_to_ms
    )
    for neuron_idx, neuron_id in enumerate(tqdm(all_neuron_ids, desc="Raster computation"))
)

all_trial_raster_matrix = np.zeros((n_trials, n_neurons, n_time_bins_ms), dtype=np.float32)
for neuron_idx, neuron_raster in raster_results:
    all_trial_raster_matrix[:, neuron_idx, :] = neuron_raster

# Helper function for PSTH matrix computation per neuron
def compute_psth_for_neuron(neuron_idx, all_trial_raster_matrix, psth_window_size_ms, n_time_bins_ms):
    """
    Compute PSTH matrix for a single neuron (using ms-based raster)
    
    Uses a sliding window approach: for each 1ms bin, calculate firing rate using
    a psth_window_size_ms (e.g., 20ms) window centered on that bin.
    
    This matches the MATLAB reference implementation:
    - Each bin represents 1ms
    - For each time point, use a sliding window of psth_window_size_ms
    - Window is centered on the time point (e.g., for 20ms window at time 50ms: 
      uses bins 40-60, which is 21 bins total, representing ~20ms centered at 50ms)
    """
    raster_raw = all_trial_raster_matrix[:, neuron_idx, :]
    neuron_psth = np.zeros((n_trials, n_time_bins_ms), dtype=np.float32)
    
    # Convert to 0-based indexing for Python (MATLAB uses 1-based)
    for time_point_ms in range(n_time_bins_ms):
        # MATLAB equivalent: time_points-psth_window_size_ms/2:time_points+psth_window_size_ms/2
        # For 20ms window at time_point_ms=50: window = 40:60 (21 bins, centered at 50)
        time_point_1based = time_point_ms + 1  # Convert to 1-based for comparison
        
        # Calculate window boundaries (matching MATLAB logic)
        if time_point_1based - psth_window_size_ms // 2 < 1:
            # Near start: use first psth_window_size_ms bins
            window_start_ms = 0
            window_end_ms = psth_window_size_ms
        elif time_point_1based + psth_window_size_ms // 2 > n_time_bins_ms:
            # Near end: use last psth_window_size_ms bins
            window_start_ms = n_time_bins_ms - psth_window_size_ms
            window_end_ms = n_time_bins_ms
        else:
            # Middle: centered window
            window_start_ms = time_point_ms - psth_window_size_ms // 2
            window_end_ms = time_point_ms + psth_window_size_ms // 2 + 1
        
        time_window = np.arange(window_start_ms, window_end_ms)
        
        # PSTH: sum spikes in window and convert to Hz (spikes per second)
        # Since raster is already in spikes per 1ms bin, we multiply by 1000 to get Hz
        # Formula: (spikes in window / window_size_ms) * 1000 = Hz
        neuron_psth[:, time_point_ms] = (
            1000 * np.sum(raster_raw[:, time_window], axis=1) / len(time_window)
        )
    
    return neuron_idx, neuron_psth

# Parallel computation of PSTH matrix
print("Computing PSTH matrix (parallelized)...")
psth_results = Parallel(n_jobs=-1, backend='threading')(
    delayed(compute_psth_for_neuron)(
        neuron_idx, all_trial_raster_matrix, psth_window_size_ms, n_time_bins_ms
    )
    for neuron_idx in tqdm(range(n_neurons), desc="PSTH computation")
)

all_trial_psth_matrix = np.zeros((n_trials, n_neurons, n_time_bins_ms), dtype=np.float32)
for neuron_idx, neuron_psth in psth_results:
    all_trial_psth_matrix[:, neuron_idx, :] = neuron_psth

print(f"Computation completed!")
print(f"Raster matrix shape: {all_trial_raster_matrix.shape} (n_trials, n_neurons, n_time_bins_ms)")
print(f"PSTH matrix shape: {all_trial_psth_matrix.shape} (n_trials, n_neurons, n_time_bins_ms)")
print(f"Time range: -{window_before_ms}ms to +{window_after_ms}ms ({total_time_ms}ms total, {n_time_bins_ms} bins at 1ms/bin)")


Computing raster matrix (parallelized, binned by milliseconds)...


Raster computation: 100%|██████████| 164/164 [01:07<00:00,  2.42it/s]


Computing PSTH matrix (parallelized)...


PSTH computation: 100%|██████████| 164/164 [00:08<00:00, 19.75it/s]


Computation completed!
Raster matrix shape: (8489, 164, 600) (n_trials, n_neurons, n_time_bins_ms)
PSTH matrix shape: (8489, 164, 600) (n_trials, n_neurons, n_time_bins_ms)
Time range: -150ms to +450ms (600ms total, 600 bins at 1ms/bin)


In [9]:
all_trial_psth_matrix_dict = {}
trial_image_dict = {}

print("按neuron筛选trials（总firing rate >= 1）...")

for neuron_idx, neuron_id in enumerate(all_neuron_ids):
    neuron_psth = all_trial_psth_matrix[:, neuron_idx, :]
    
    total_firing_rates = np.sum(neuron_psth, axis=1)
    
    valid_trial_mask = total_firing_rates >= 20.0
    valid_trial_indices = np.where(valid_trial_mask)[0]
    
    filtered_psth = neuron_psth[valid_trial_indices, :]
    filtered_images = stimulus_ids[valid_trial_indices]

    if filtered_psth.shape[0] >= 1000:
        all_trial_psth_matrix_dict[neuron_id] = filtered_psth
        trial_image_dict[neuron_id] = filtered_images
    

print(f"\n筛选完成！")
print(f"  PSTH形状: {all_trial_psth_matrix_dict[all_neuron_ids[1]].shape}")


按neuron筛选trials（总firing rate >= 1）...

筛选完成！
  PSTH形状: (4444, 600)


In [10]:
from scipy.stats import ranksums

baseline_start_ms = -30
baseline_end_ms = 30
response_window1_start_ms = 50
response_window1_end_ms = 120
response_window2_start_ms = 120
response_window2_end_ms = 240

# Re-define window_before_ms (raster matrix is now binned by milliseconds)
window_before_ms = 150  # milliseconds before stimulus

# Convert time windows to raster matrix indices (raster is binned by milliseconds: 1ms per bin)
# Index = relative_time_ms + window_before_ms
baseline_start_idx = int(baseline_start_ms + window_before_ms)
baseline_end_idx = int(baseline_end_ms + window_before_ms) + 1
response1_start_idx = int(response_window1_start_ms + window_before_ms)
response1_end_idx = int(response_window1_end_ms + window_before_ms) + 1
response2_start_idx = int(response_window2_start_ms + window_before_ms)
response2_end_idx = int(response_window2_end_ms + window_before_ms) + 1

if baseline_start_idx >= baseline_end_idx:
    baseline_start_idx, baseline_end_idx = baseline_end_idx, baseline_start_idx + 1

print(f"时间窗口索引 (相对于 raster 矩阵，单位: ms bins):")
print(f"  Baseline: {baseline_start_idx}-{baseline_end_idx} ({baseline_start_ms} to {baseline_end_ms} ms)")
print(f"  Response window 1: {response1_start_idx}-{response1_end_idx} ({response_window1_start_ms} to {response_window1_end_ms} ms)")
print(f"  Response window 2: {response2_start_idx}-{response2_end_idx} ({response_window2_start_ms} to {response_window2_end_ms} ms)")

all_trial_psth_matrix_dict_filtered = {}
trial_image_dict_filtered = {}

print(f"\n进行Wilcoxon rank-sum test识别响应神经元...")
print(f"筛选前neuron数量: {len(all_trial_psth_matrix_dict)}")

for neuron_idx, neuron_id in enumerate(all_neuron_ids):
    if neuron_id not in all_trial_psth_matrix_dict:
        continue
    
    neuron_psth = all_trial_psth_matrix_dict[neuron_id]
    neuron_raster = all_trial_raster_matrix[:, neuron_idx, :]
    
    neuron_images = trial_image_dict[neuron_id]
    valid_trial_indices = np.where(np.isin(stimulus_ids, neuron_images))[0]
    
    if len(valid_trial_indices) == 0:
        continue
    
    neuron_raster_filtered = neuron_raster[valid_trial_indices, :]
    
    baseline_firing_rates = np.mean(neuron_raster_filtered[:, baseline_start_idx:baseline_end_idx], axis=1)
    response1_firing_rates = np.mean(neuron_raster_filtered[:, response1_start_idx:response1_end_idx], axis=1)
    response2_firing_rates = np.mean(neuron_raster_filtered[:, response2_start_idx:response2_end_idx], axis=1)
    
    stat1, p_val1 = ranksums(baseline_firing_rates, response1_firing_rates, alternative='two-sided')
    stat2, p_val2 = ranksums(baseline_firing_rates, response2_firing_rates, alternative='two-sided')
    
    min_p_val = min(p_val1, p_val2)
    
    if min_p_val < 0.001:
        all_trial_psth_matrix_dict_filtered[neuron_id] = neuron_psth
        trial_image_dict_filtered[neuron_id] = neuron_images

print(f"筛选后neuron数量: {len(all_trial_psth_matrix_dict_filtered)}")
print(f"保留的neuron比例: {len(all_trial_psth_matrix_dict_filtered)/len(all_trial_psth_matrix_dict)*100:.1f}%")

all_trial_psth_matrix_dict = all_trial_psth_matrix_dict_filtered
trial_image_dict = trial_image_dict_filtered


时间窗口索引 (相对于 raster 矩阵，单位: ms bins):
  Baseline: 120-181 (-30 to 30 ms)
  Response window 1: 200-271 (50 to 120 ms)
  Response window 2: 270-391 (120 to 240 ms)

进行Wilcoxon rank-sum test识别响应神经元...
筛选前neuron数量: 164
筛选后neuron数量: 131
保留的neuron比例: 79.9%


In [ ]:
# def give_me_nc(raster, img_idx, interested_img, boot_times=5):
#     """
#     计算 reliability (split-half correlation)
    
#     参数:
#     raster: 每个 trial 的平均响应 (n_trials,)
#     img_idx: 每个 trial 对应的图像名称 (n_trials,)
#     interested_img: 感兴趣的图像列表
#     boot_times: bootstrap 次数
    
#     返回:
#     r: reliability 值
#     """
#     from scipy.stats import pearsonr
    
#     Image_LOCs = {}
#     for ii in interested_img:
#         Image_LOCs[ii] = np.where(img_idx == ii)[0]
    
#     d1 = np.zeros(len(interested_img))
#     d2 = np.zeros(len(interested_img))
#     r = np.zeros(boot_times)
    
#     for bb in range(boot_times):
#         for idx, ii in enumerate(interested_img):
#             LOC_NOW = Image_LOCs[ii]
#             data_length = len(LOC_NOW)
#             half_points = data_length // 2
            
#             if data_length < 2:
#                 d1[idx] = 0
#                 d2[idx] = 0
#                 continue
            
#             Order_now = np.random.permutation(data_length)
#             first_half = LOC_NOW[Order_now[:half_points]]
#             second_half = LOC_NOW[Order_now[half_points:]]
            
#             d1[idx] = np.mean(raster[first_half])
#             d2[idx] = np.mean(raster[second_half])
        
#         if len(interested_img) > 1:
#             r[bb] = pearsonr(d1, d2)[0]
#         else:
#             r[bb] = 0.0
    
#     r_mean = np.mean(r)
#     if np.isnan(r_mean) or r_mean <= 0:
#         return 0.0
    
#     r_corrected = (2 * r_mean) / (1 + r_mean)
#     return r_corrected

# print("动态搜索最佳时间窗口并计算 reliability...")

# # Re-define window_before_ms (raster matrix is now binned by milliseconds)
# window_before_ms = 150  # milliseconds before stimulus

# basic_time_bin_ms = np.arange(70, 221)
# # Convert to raster matrix indices (raster is binned by milliseconds: 1ms per bin)
# # Index = time_ms + window_before_ms
# basic_time_bin_indices = (basic_time_bin_ms + window_before_ms).astype(int)

# bin_1_ms = np.arange(20, 201, 10)
# bin_2_ms = np.arange(90, 391, 10)

# window_summary = []
# for b1 in bin_1_ms:
#     for b2 in bin_2_ms:
#         if b1 < b2:
#             window_summary.append([b1, b2])
#         else:
#             window_summary.append([b2, b1])

# window_summary = np.array(window_summary)
# bin_size = len(window_summary)

# random_order = np.random.permutation(1000)
# find_sample = random_order[:500]
# test_sample = random_order[500:1000]

# reliability_dict = {}
# reliability_basic_dict = {}
# best_r_time1_dict = {}
# best_r_time2_dict = {}
# r_search_pool_dict = {}

# for neuron_id in list(all_trial_psth_matrix_dict.keys()):
#     neuron_psth = all_trial_psth_matrix_dict[neuron_id]
#     neuron_images = trial_image_dict[neuron_id]
    
#     valid_basic_indices = basic_time_bin_indices[(basic_time_bin_indices >= 0) & (basic_time_bin_indices < neuron_psth.shape[1])]
#     if len(valid_basic_indices) == 0:
#         reliability_dict[neuron_id] = 0.0
#         reliability_basic_dict[neuron_id] = 0.0
#         best_r_time1_dict[neuron_id] = 70
#         best_r_time2_dict[neuron_id] = 220
#         continue
    
#     basic_raster = np.mean(neuron_psth[:, valid_basic_indices], axis=1)
#     test_images = stimulus_unique[test_sample] if len(stimulus_unique) >= 1000 else stimulus_unique
#     reliability_basic_dict[neuron_id] = give_me_nc(
#         basic_raster, 
#         neuron_images, 
#         test_images, 
#         boot_times=5
#     )
    
#     single_r_pool = np.zeros(bin_size)
    
#     for bin_idx in range(bin_size):
#         t1_ms = window_summary[bin_idx, 0]
#         t2_ms = window_summary[bin_idx, 1]
        
#         # Convert time windows to raster matrix indices (raster is binned by milliseconds: 1ms per bin)
#         # Index = time_ms + window_before_ms
#         t1_indices = int(t1_ms) + window_before_ms
#         t2_indices = int(t2_ms) + window_before_ms
        
#         valid_indices = np.arange(t1_indices, t2_indices + 1)
#         valid_indices = valid_indices[(valid_indices >= 0) & (valid_indices < neuron_psth.shape[1])]
        
#         if len(valid_indices) == 0:
#             single_r_pool[bin_idx] = 0.0
#             continue
        
#         selected_raster = np.mean(neuron_psth[:, valid_indices], axis=1)
#         find_images = stimulus_unique[find_sample] if len(stimulus_unique) >= 1000 else stimulus_unique
#         single_r_pool[bin_idx] = give_me_nc(
#             selected_raster,
#             neuron_images,
#             find_images,
#             boot_times=5
#         )
    
#     r_search_pool_dict[neuron_id] = single_r_pool
#     best_bin_idx = np.argmax(single_r_pool)
    
#     best_r_time1_dict[neuron_id] = window_summary[best_bin_idx, 0]
#     best_r_time2_dict[neuron_id] = window_summary[best_bin_idx, 1]
    
#     # Convert best time windows to raster matrix indices (raster is binned by milliseconds: 1ms per bin)
#     # Index = time_ms + window_before_ms
#     best_t1_indices = int(best_r_time1_dict[neuron_id]) + window_before_ms
#     best_t2_indices = int(best_r_time2_dict[neuron_id]) + window_before_ms
    
#     best_valid_indices = np.arange(best_t1_indices, best_t2_indices + 1)
#     best_valid_indices = best_valid_indices[(best_valid_indices >= 0) & (best_valid_indices < neuron_psth.shape[1])]
    
#     if len(best_valid_indices) > 0:
#         selected_raster_best = np.mean(neuron_psth[:, best_valid_indices], axis=1)
#         all_images = stimulus_unique[:1000] if len(stimulus_unique) >= 1000 else stimulus_unique
#         reliability_dict[neuron_id] = give_me_nc(
#             selected_raster_best,
#             neuron_images,
#             all_images,
#             boot_times=5
#         )
#     else:
#         reliability_dict[neuron_id] = 0.0

# print(f"Reliability 计算完成！")
# print(f"计算了 {len(reliability_dict)} 个神经元的 reliability")
# reliability_values = np.array(list(reliability_dict.values()))
# reliability_basic_values = np.array(list(reliability_basic_dict.values()))
# print(f"Reliability (best window) 范围: {reliability_values.min():.3f} - {reliability_values.max():.3f}")
# print(f"Reliability (best window) 均值: {reliability_values.mean():.3f} ± {reliability_values.std():.3f}")
# print(f"Reliability (basic 70-220ms) 均值: {reliability_basic_values.mean():.3f} ± {reliability_basic_values.std():.3f}")
# print(f"Reliability > 0.4 的神经元数量: {np.sum(reliability_values > 0.4)}")
# print(f"\n最佳时间窗口统计:")
# best_time1_values = np.array(list(best_r_time1_dict.values()))
# best_time2_values = np.array(list(best_r_time2_dict.values()))
# print(f"  最佳窗口开始时间: {best_time1_values.mean():.1f} ± {best_time1_values.std():.1f} ms")
# print(f"  最佳窗口结束时间: {best_time2_values.mean():.1f} ± {best_time2_values.std():.1f} ms")


动态搜索最佳时间窗口并计算 reliability...
Reliability 计算完成！
计算了 125 个神经元的 reliability
Reliability (best window) 范围: 0.000 - 0.598
Reliability (best window) 均值: 0.205 ± 0.158
Reliability (basic 70-220ms) 均值: 0.086 ± 0.099
Reliability > 0.4 的神经元数量: 19

最佳时间窗口统计:
  最佳窗口开始时间: 113.4 ± 59.8 ms
  最佳窗口结束时间: 288.7 ± 109.7 ms


In [ ]:
filtered_neuron_ids = sorted(all_trial_psth_matrix_dict.keys())
n_filtered_neurons = len(filtered_neuron_ids)
n_images = len(stimulus_unique)
n_time_bins = all_trial_psth_matrix_dict[filtered_neuron_ids[0]].shape[1]

neuron_image_response_matrix = np.zeros((n_filtered_neurons, n_images, n_time_bins), dtype=np.float32)
image_to_index = {img: idx for idx, img in enumerate(stimulus_unique)}

for neuron_idx, neuron_id in enumerate(filtered_neuron_ids):
    neuron_psth = all_trial_psth_matrix_dict[neuron_id]
    neuron_images = trial_image_dict[neuron_id]
    
    for image_name in stimulus_unique:
        image_idx = image_to_index[image_name]
        image_mask = neuron_images == image_name
        image_trials = neuron_psth[image_mask, :]
        
        if len(image_trials) > 0:
            neuron_image_response_matrix[neuron_idx, image_idx, :] = np.mean(image_trials, axis=0)
    

print(f"\n响应矩阵形状: {neuron_image_response_matrix.shape}")



响应矩阵形状: (131, 1000, 600)


In [65]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from PIL import Image
import open_clip

class TemporalEPEncoder(nn.Module):
    def __init__(self, input_dim=244, time_bins=30, d_model=256, n_token=128, 
                 num_conv_layers=3, dropout=0.2, output_dim=768):
        super().__init__()
        self.input_dim = input_dim
        self.time_bins = time_bins
        self.d_model = d_model
        self.n_token = n_token
        self.output_dim = output_dim
        
        if input_dim > 200:
            hidden_dim = min(input_dim // 4, d_model * 8)
            self.input_proj = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, d_model * 4),
                nn.LayerNorm(d_model * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
            )
        else:
            self.input_proj = nn.Sequential(
                nn.Linear(input_dim, d_model * 4),
                nn.LayerNorm(d_model * 4),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(d_model * 4, d_model),
            )
        
        conv_layers = []
        for i in range(num_conv_layers):
            if i == 0:
                in_channels = d_model
            else:
                in_channels = d_model * 2
            
            if i == num_conv_layers - 1:
                out_channels = d_model
            else:
                out_channels = d_model * 2
            
            conv_layers.extend([
                nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.BatchNorm1d(out_channels),
                nn.GELU(),
                nn.Dropout(dropout)
            ])
        self.temporal_conv = nn.Sequential(*conv_layers)
        
        self.adaptive_pool = nn.AdaptiveAvgPool1d(n_token)
        
        self.final_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout * 0.5)
        )
        
        self.feature_proj = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.LayerNorm(d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, output_dim),
            nn.LayerNorm(output_dim)
        )
        
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm1d, nn.LayerNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        B = x.shape[0]
        
        if torch.isnan(x).any() or torch.isinf(x).any():
            x = torch.nan_to_num(x, nan=0.0, posinf=1.0, neginf=-1.0)
        
        x = self.input_proj(x)
        x = x.transpose(1, 2)
        x = self.temporal_conv(x)
        x = self.adaptive_pool(x)
        x = x.transpose(1, 2)
        x = self.final_proj(x)
        
        x = self.feature_proj(x)
        feature = x.mean(dim=1)
        
        return feature


class PSTHClusterClassifier(nn.Module):
    def __init__(self, encoder, num_classes=7):
        super().__init__()
        self.encoder = encoder
        feat_dim = getattr(encoder, 'output_dim', 768)
        self.classifier = nn.Linear(feat_dim, num_classes)
    def forward(self, x):
        feat = self.encoder(x)
        return self.classifier(feat)



In [62]:
from typing import Any


class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
        
    def forward(self, features_1, features_2):
        features_1 = features_1.float()
        features_2 = features_2.float()
        
        features_1 = F.normalize(features_1, dim=1)
        features_2 = F.normalize(features_2, dim=1)
        
        logits_12 = torch.matmul(features_1, features_2.T) / self.temperature
        logits_21 = torch.matmul(features_2, features_1.T) / self.temperature
        
        batch_size = features_1.size(0)
        labels = torch.arange(batch_size, device=features_1.device)
        
        loss_12 = F.cross_entropy(logits_12, labels)
        loss_21 = F.cross_entropy(logits_21, labels)
        
        return (loss_12 + loss_21) / 2

class PSTHDataset(Dataset):
    def __init__(self, psth_data, image_paths, image_names, clip_model, clip_preprocess, device='cpu'):
        self.psth_data = torch.tensor(psth_data, dtype=torch.float32)
        self.image_paths = image_paths
        self.image_names = image_names
        self.clip_model = clip_model
        self.clip_preprocess = clip_preprocess
        self.device = device
        
        self.clip_features_cache = {}
        self._precompute_clip_features()
    
    def _precompute_clip_features(self):
        print("预计算CLIP特征...")
        self.clip_model.eval()
        with torch.no_grad():
            for idx, img_path in enumerate[Any](self.image_paths):
                if idx % 100 == 0:
                    print(f"处理进度: {idx}/{len(self.image_paths)}")
                try:
                    img = Image.open(img_path).convert('RGB')
                    img_tensor = self.clip_preprocess(img).unsqueeze(0).to(self.device)
                    clip_dtype = next(self.clip_model.parameters()).dtype
                    img_tensor = img_tensor.to(clip_dtype)
                    clip_feature = self.clip_model.encode_image(img_tensor)
                    clip_feature = F.normalize(clip_feature, dim=-1)
                    self.clip_features_cache[idx] = clip_feature.cpu()
                except Exception as e:
                    print(f"Error loading image {img_path}: {e}")
                    self.clip_features_cache[idx] = torch.zeros(1, 768)
        print("CLIP特征预计算完成")
    
    def __len__(self):
        return len(self.psth_data)
    
    def __getitem__(self, idx):
        psth = self.psth_data[idx]
        psth = psth.transpose(0, 1)
        clip_feature = self.clip_features_cache[idx].squeeze(0)
        return psth, clip_feature


class PSTHClusterDataset(Dataset):
    def __init__(self, psth_data, cluster_labels):
        self.psth_data = torch.tensor(psth_data, dtype=torch.float32)
        self.cluster_labels = np.array(cluster_labels, dtype=np.int64)
    def __len__(self):
        return len(self.psth_data)
    def __getitem__(self, idx):
        psth = self.psth_data[idx].transpose(0, 1)
        return psth, self.cluster_labels[idx]


In [63]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")

PKL_PATH = "/media/ubuntu/sda/Monkey/semantic/epoch_10_monkeyN/merged_cluster_class_counts.pkl"
with open(PKL_PATH, 'rb') as f:
    cluster_data = pickle.load(f)
class_to_cluster = {}
for cluster_id in range(7):
    for class_name in cluster_data[cluster_id]:
        class_to_cluster[class_name] = cluster_id

condition_df = pd.read_csv("/media/ubuntu/sda/duan/result/260121/images_sequence_10000.csv")
image_path_dict = {row['image_name']: row['image_path'] for _, row in condition_df.iterrows()}

valid_psth = []
valid_labels = []

for stim_idx, img_name in enumerate(stimulus_unique):
    if img_name not in image_path_dict:
        continue
    path = image_path_dict[img_name]
    class_name = os.path.basename(os.path.dirname(path))
    if class_name not in class_to_cluster:
        continue
    psth_per_image = neuron_image_response_matrix[:, stim_idx, 150:450]
    if psth_per_image.sum() <= 0:
        continue
    valid_psth.append(psth_per_image)
    valid_labels.append(class_to_cluster[class_name])

valid_psth = np.array(valid_psth)
valid_labels = np.array(valid_labels)
print(f"有效数据(7类cluster): {len(valid_psth)} 张图片")
print(f"PSTH形状: {valid_psth.shape}, 各类数量: {np.bincount(valid_labels, minlength=7)}")

train_indices, val_indices = train_test_split(
    range(len(valid_psth)),
    test_size=0.1,
    random_state=42,
    stratify=valid_labels
)

train_psth = valid_psth[train_indices]
train_labels = valid_labels[train_indices]
val_psth = valid_psth[val_indices]
val_labels = valid_labels[val_indices]

print(f"训练集: {len(train_psth)}, 验证集: {len(val_psth)}")

train_dataset = PSTHClusterDataset(train_psth, train_labels)
val_dataset = PSTHClusterDataset(val_psth, val_labels)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)



使用设备: cuda
有效数据(7类cluster): 1000 张图片
PSTH形状: (1000, 131, 300), 各类数量: [ 28 176  32  95 195 217 257]
训练集: 900, 验证集: 100


In [66]:
n_neurons = train_psth.shape[1]
n_time_bins = train_psth.shape[2]

encoder_backbone = TemporalEPEncoder(
    input_dim=n_neurons,
    time_bins=n_time_bins,
    d_model=64,
    n_token=128,
    num_conv_layers=2,
    dropout=0.2,
    output_dim=768
)
model = PSTHClusterClassifier(encoder_backbone, num_classes=7).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

num_epochs = 10
best_val_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for psth_batch, label_batch in train_loader:
        psth_batch = psth_batch.to(device)
        label_batch = label_batch.to(device)

        optimizer.zero_grad()
        logits = model(psth_batch)
        loss = criterion(logits, label_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * psth_batch.size(0)
        pred = logits.argmax(dim=1)
        train_correct += (pred == label_batch).sum().item()
        train_total += label_batch.size(0)

    avg_train_loss = train_loss / train_total
    train_acc = train_correct / train_total
    scheduler.step()

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for psth_batch, label_batch in val_loader:
            psth_batch = psth_batch.to(device)
            label_batch = label_batch.to(device)
            logits = model(psth_batch)
            loss = criterion(logits, label_batch)
            val_loss += loss.item() * psth_batch.size(0)
            pred = logits.argmax(dim=1)
            val_correct += (pred == label_batch).sum().item()
            val_total += label_batch.size(0)

    avg_val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.4f}, Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{output_dir}/psth_cluster_classifier_best.pth')
        print(f"保存最佳模型，验证准确率: {best_val_acc:.4f}")

print("训练完成！")



Epoch 1/10 - Train Loss: 1.7760 Acc: 0.2322, Val Loss: 1.7441 Acc: 0.2200
保存最佳模型，验证准确率: 0.2200
Epoch 2/10 - Train Loss: 1.7273 Acc: 0.2889, Val Loss: 1.7490 Acc: 0.2800
保存最佳模型，验证准确率: 0.2800
Epoch 3/10 - Train Loss: 1.7056 Acc: 0.2933, Val Loss: 1.7276 Acc: 0.2500
Epoch 4/10 - Train Loss: 1.6712 Acc: 0.3033, Val Loss: 1.7502 Acc: 0.2600
Epoch 5/10 - Train Loss: 1.5992 Acc: 0.3911, Val Loss: 1.7550 Acc: 0.2300
Epoch 6/10 - Train Loss: 1.5192 Acc: 0.4589, Val Loss: 1.8345 Acc: 0.2300
Epoch 7/10 - Train Loss: 1.4492 Acc: 0.4722, Val Loss: 1.9128 Acc: 0.2500
Epoch 8/10 - Train Loss: 1.3576 Acc: 0.5300, Val Loss: 1.8938 Acc: 0.2300
Epoch 9/10 - Train Loss: 1.2825 Acc: 0.5689, Val Loss: 1.9919 Acc: 0.2600
Epoch 10/10 - Train Loss: 1.1955 Acc: 0.5911, Val Loss: 2.0557 Acc: 0.2600
训练完成！


In [ ]:
class PSTHClusterDataset(Dataset):
    def __init__(self, psth_data, cluster_labels):
        self.psth_data = torch.tensor(psth_data, dtype=torch.float32)
        self.cluster_labels = np.array(cluster_labels, dtype=np.int64)
    def __len__(self):
        return len(self.psth_data)
    def __getitem__(self, idx):
        psth = self.psth_data[idx].transpose(0, 1)
        return psth, self.cluster_labels[idx]

In [154]:
encoder.load_state_dict(torch.load(f'{output_dir}/temporal_ep_encoder_best.pth'))
encoder.eval()

test_features = []
test_image_paths = []

with torch.no_grad():
    for batch_idx, (psth_batch, clip_batch) in enumerate(val_loader):
        psth_batch = psth_batch.to(device)
        
        ep_features = encoder(psth_batch)
        test_features.append(ep_features.cpu())
        
        start_idx = batch_idx * val_loader.batch_size
        end_idx = min(start_idx + val_loader.batch_size, len(val_dataset.image_paths))
        test_image_paths.extend(val_dataset.image_paths[start_idx:end_idx])

test_features = torch.cat(test_features, dim=0).numpy()
test_image_paths = np.array(test_image_paths)


np.save(f'{output_dir}/test_features.npy', test_features)
np.save(f'{output_dir}/test_image_paths.npy', test_image_paths)

print(f"\n✓ 特征矩阵已保存到: {output_dir}/test_features.npy (shape: {test_features.shape})")
print(f"✓ 图像路径已保存到: {output_dir}/test_image_paths.npy (shape: {test_image_paths.shape})")




✓ 特征矩阵已保存到: /media/ubuntu/sda/duan/result/260121/test_features.npy (shape: (100, 768))
✓ 图像路径已保存到: /media/ubuntu/sda/duan/result/260121/test_image_paths.npy (shape: (100,))
